In [1]:
import pandas as pd
import numpy as np
import random

In [3]:
lpi = pd.read_csv('/content/International_LPI_Scorecard_Select_Countries_v2.csv')
combined_ppi = pd.read_csv('/content/combined_ppi_data.csv')
commodity_prices = pd.read_csv('/content/monthly_world_bank_commodity_prices_1960_2025_v2.csv')
tariff = pd.read_csv('/content/tarriff_database_2025_only_semiconductor_components_v2.csv')

In [ ]:
viable_countries = lpi.country_code.unique()

In [ ]:
viable_countries

array(['AUS', 'BEL', 'BRA', 'CAN', 'CHN', 'FIN', 'FRA', 'DEU', 'HKG',
       'IND', 'IDN', 'JPN', 'MYS', 'MEX', 'NLD', 'SGP', 'THA', 'ARE',
       'GBR', 'USA'], dtype=object)

In [ ]:
sources = combined_ppi.country_code.unique()

In [ ]:
sources

array(['USA', 'CAN', 'CHN', 'MEX', 'OAC'], dtype=object)

In [ ]:
combined_ppi.head()

,date,year,month,ppi_value,series_id,industry,product,source,country_code,base_year
0,1981-06-01,1981,6,181.0,PCU334413334413P,Semiconductor and related device manufacturing,Primary products,BLS,USA,1998
1,1981-07-01,1981,7,180.5,PCU334413334413P,Semiconductor and related device manufacturing,Primary products,BLS,USA,1998
2,1981-08-01,1981,8,180.8,PCU334413334413P,Semiconductor and related device manufacturing,Primary products,BLS,USA,1998
3,1981-09-01,1981,9,179.5,PCU334413334413P,Semiconductor and related device manufacturing,Primary products,BLS,USA,1998
4,1981-10-01,1981,10,180.0,PCU334413334413P,Semiconductor and related device manufacturing,Primary products,BLS,USA,1998


In [ ]:
combined_ppi.drop(combined_ppi.loc[lambda x: x.source == 'BLS'].index, inplace = True) # new

# Base Years for FRED data

In [ ]:
combined_ppi.country_code.unique()

array(['CAN', 'CHN', 'MEX', 'OAC', 'USA'], dtype=object)

In [ ]:
combined_ppi.loc[lambda x: x.country_code =='CAN'].base_year.unique()

array([2012])

In [ ]:
combined_ppi.loc[lambda x: x.country_code =='USA'].base_year.unique()

array([2005])

In [ ]:
combined_ppi.loc[lambda x: x.country_code =='OAC'].base_year.unique()

array([2012])

In [ ]:
combined_ppi.loc[lambda x: x.country_code =='CHN'].base_year.unique()

array([2012])

In [ ]:
combined_ppi.loc[lambda x: x.country_code =='MEX'].base_year.unique()

array([2012])

# Ending Years for Each Countries (FRED) PPI

In [ ]:
combined_ppi.groupby('country_code').date.max()

,date
country_code,
CAN,2020-12-01
CHN,2018-12-01
MEX,2025-12-01
OAC,2025-12-01
USA,2025-12-01


# Extracting Unique Country Codes


In [ ]:
all_unique_countries = set(lpi['country_code'].unique())

unique_countries = sorted(list(all_unique_countries))
print(f"Total unique countries identified: {len(unique_countries)}")
print(f"Unique countries: {unique_countries}")

Total unique countries identified: 20
Unique countries: ['ARE', 'AUS', 'BEL', 'BRA', 'CAN', 'CHN', 'DEU', 'FIN', 'FRA', 'GBR', 'HKG', 'IDN', 'IND', 'JPN', 'MEX', 'MYS', 'NLD', 'SGP', 'THA', 'USA']


# Mapping

## Integrated Circuit Parts

Creating synthetic wholesale per-unit price for an integrated circuit component

Will use 2012 as baseline year since all countries other than FRED USA begin in 2012.

The US FRED PPI begins in 2005.

Will rebase the FRED USA series to 2012 (so PPI value is 100 for that year).



In [ ]:
# country-specific 2012 baseline wholesale starting prices
# represents cross-country structural cost differences
baseline_circuits = {
    'USA': 0.20, # mature, high-cost manufacturing
    'JPN': 0.21, # high labor, high quality, stock packaging ecosystem
    'DEU': 0.20, # Germany; Slightly below Japan, advanced high-cost EU manufacturing
    'FRA': 0.20, # Similar to DEU; Western Europe cost structure
    'GBR': 0.20, # Similar Western Europe cost level
    'NLD': 0.20, # Western Europe, high-cost but efficient
    'BEL': 0.20, # Western Europe, similar to FRA/NLD
    'CAN': 0.18, # Slightly below US; similar structure, marginally lower costs
    'AUS': 0.18, # Developed, smaller electronics base, similar to CAN
    'FIN': 0.20, # Nordic, high labor cost, niche electronics
    'SGP': 0.18, # Advanced hub, efficient, slightly cheaper than US
    'HKG': 0.17, # Trade/assembly hub, lower labor than US/EU
    'MYS': 0.13, # strong electronics assembly
    'CHN': 0.12, # Very low labor + scale advantages
    'MEX': 0.15, # Nearshore, lower labor than US, higher than China/MYS
    'BRA': 0.16, # Higher friction than Asia, but lower labor than US/EU
    'IND': 0.14, # Low labor, less mature packaging ecosystem
    'THA': 0.14, # similar reasons
    'IDN': 0.13, # Low labor, emerging electronics base
    'ARE': 0.20, # UAE; High-income, imported tech, niche local assembly
}

## Mapping Countries to PPI tables

In [ ]:
ppi_map = {
    'USA': ('FRED', 'USA'),
    'CAN': ('FRED', 'CAN'),
    'CHN': ('FRED', 'CHN'),
    'MEX': ('FRED', 'MEX'),
    'OAC': ('FRED', 'OAC'),

    # Countries mapped to OAC (Asian import PPI)
    'ARE': ('FRED', 'OAC'), # Gulf imports dominated by Asian electronics
    'AUS': ('FRED', 'OAC'), # Asia-Pacific supply chain
    'HKG': ('FRED', 'OAC'), # Part of Asian supply chain
    'IDN': ('FRED', 'OAC'), # Southeast Asia
    'IND': ('FRED', 'OAC'), # South Asia, similar to OAC basket
    'JPN': ('FRED', 'OAC'), # Part of Asian electronics ecosystem
    'MYS': ('FRED', 'OAC'),
    'SGP': ('FRED', 'OAC'),
    'THA': ('FRED', 'OAC'),

    # Countries mapped to FRED USA (global import PPI)
    # U.S. import PPI is closest match to European semiconductor cost dymanics
    'BEL': ('FRED', 'USA'), # Western Europe -> Similar inflation to US imprt
    'DEU': ('FRED', 'USA'), # EU inflation similar to US import PPIs
    'FIN': ('FRED', 'USA'), # EU electronics inflation similar to US
    'FRA': ('FRED', 'USA'),
    'GBR': ('FRED', 'USA'),
    'NLD': ('FRED', 'USA'),

    # Brazil still maps to OAC
    'BRA': ('FRED', 'OAC') # Brazil imports Asian semiconductors
}


## Adding PPI source and region columns based on country mapping

Tell model which PPI series to reference for scaling

In [ ]:
combined_ppi['ppi_source'] = combined_ppi['country_code'].map(
    lambda c: ppi_map[c][0]
)
combined_ppi['ppi_region'] = combined_ppi['country_code'].map(
    lambda c: ppi_map[c][1]
)

## Determining earliest start date for each ppi_source & ppi_region

In [ ]:
# finds first availale month for each PPI series
start_dates = (
    combined_ppi
    .groupby(['source', 'country_code'])['date']
    .min()
)


In [ ]:
start_dates

source  country_code
FRED    CAN             2012-06-01
        CHN             2012-06-01
        MEX             2012-06-01
        OAC             2012-06-01
        USA             2005-12-01
Name: date, dtype: object

# Rebasing FRED USA series so that 2012 = ~ 100

- CAN/CHN/MEX/OAC are anchored to 2012
- USA is currently anchored to 2005

In [ ]:
combined_ppi.loc[lambda x: (x.source == 'FRED') & (x.country_code == 'USA') & (x.year == 2012)].head()

,date,year,month,ppi_value,series_id,industry,product,source,country_code,base_year,ppi_source,ppi_region
2686,2012-01-01,2012,1,84.0,IZ3344,Semiconductor and other electronic component m...,Semiconductor and other electronic components,FRED,USA,2005,FRED,USA
2687,2012-02-01,2012,2,83.8,IZ3344,Semiconductor and other electronic component m...,Semiconductor and other electronic components,FRED,USA,2005,FRED,USA
2688,2012-03-01,2012,3,83.7,IZ3344,Semiconductor and other electronic component m...,Semiconductor and other electronic components,FRED,USA,2005,FRED,USA
2689,2012-04-01,2012,4,82.8,IZ3344,Semiconductor and other electronic component m...,Semiconductor and other electronic components,FRED,USA,2005,FRED,USA
2690,2012-05-01,2012,5,82.7,IZ3344,Semiconductor and other electronic component m...,Semiconductor and other electronic components,FRED,USA,2005,FRED,USA


In [ ]:
ppi_usa_2012 = (
    combined_ppi
    .query("source == 'FRED' and country_code == 'USA' and year == 2012")
    ['ppi_value']
    .mean()
)


In [ ]:
ppi_usa_2012

np.float64(82.46666666666665)

In [ ]:
# rebasing full USA series
combined_ppi.loc[lambda x:
 (x.source == 'FRED') & (x.country_code == 'USA'),
                 'ppi_value'] = (
                     combined_ppi.loc[
                         lambda x: (x.source == 'FRED') & (x.country_code == 'USA'),
                         'ppi_value'] / ppi_usa_2012) * 100


In [ ]:
# building a 2012 anchor table
ppi_anchor_2012 = (
    combined_ppi
    .query("source == 'FRED' and year == 2012")
    .groupby(['source', 'country_code'])['ppi_value']
    .mean()
)

In [ ]:
ppi_anchor_2012

source  country_code
FRED    CAN              99.800000
        CHN             100.371429
        MEX              99.542857
        OAC              98.571429
        USA             100.000000
Name: ppi_value, dtype: float64

In [ ]:
combined_ppi.loc[lambda x: (x.source == 'FRED') & (x.country_code == 'CHN')].tail()

,date,year,month,ppi_value,series_id,industry,product,source,country_code,base_year,ppi_source,ppi_region
2282,2018-08-01,2018,8,89.0,COCHNZ3344,Semiconductor and other electronic component m...,Semiconductor and other electronic components,FRED,CHN,2012,FRED,CHN
2283,2018-09-01,2018,9,89.0,COCHNZ3344,Semiconductor and other electronic component m...,Semiconductor and other electronic components,FRED,CHN,2012,FRED,CHN
2284,2018-10-01,2018,10,89.1,COCHNZ3344,Semiconductor and other electronic component m...,Semiconductor and other electronic components,FRED,CHN,2012,FRED,CHN
2285,2018-11-01,2018,11,89.4,COCHNZ3344,Semiconductor and other electronic component m...,Semiconductor and other electronic components,FRED,CHN,2012,FRED,CHN
2286,2018-12-01,2018,12,89.4,COCHNZ3344,Semiconductor and other electronic component m...,Semiconductor and other electronic components,FRED,CHN,2012,FRED,CHN


In [ ]:
combined_ppi.loc[lambda x: (x.source == 'FRED') & (x.country_code == 'CAN')].tail()

,date,year,month,ppi_value,series_id,industry,product,source,country_code,base_year,ppi_source,ppi_region
2203,2020-08-01,2020,8,94.0,COCANZ334,Computer and electronic product manufacturing,Computer and electronic manufacturing,FRED,CAN,2012,FRED,CAN
2204,2020-09-01,2020,9,93.6,COCANZ334,Computer and electronic product manufacturing,Computer and electronic manufacturing,FRED,CAN,2012,FRED,CAN
2205,2020-10-01,2020,10,93.4,COCANZ334,Computer and electronic product manufacturing,Computer and electronic manufacturing,FRED,CAN,2012,FRED,CAN
2206,2020-11-01,2020,11,93.4,COCANZ334,Computer and electronic product manufacturing,Computer and electronic manufacturing,FRED,CAN,2012,FRED,CAN
2207,2020-12-01,2020,12,93.4,COCANZ334,Computer and electronic product manufacturing,Computer and electronic manufacturing,FRED,CAN,2012,FRED,CAN


# Filling Missing PPI Values for CAN and CHN

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

def extend_ppi_tail(group: pd.DataFrame, target_end: str = "2025-12-01") -> pd.DataFrame:
    group = group.sort_values("date").copy()
    group["date"] = pd.to_datetime(group["date"])
    group = group.set_index("date")

    full_idx = pd.date_range(group.index.min(), pd.Timestamp(target_end), freq="MS")
    y = group["ppi_value"].reindex(full_idx)

    # Keep observed history intact
    observed = y.dropna()

    # Nothing to extend
    if observed.empty or observed.index.max() >= pd.Timestamp(target_end):
        out = y.ffill()
    else:
        last_obs_date = observed.index.max()
        steps = len(pd.date_range(last_obs_date, pd.Timestamp(target_end), freq="MS")) - 1

        # Mild fallback if too little data
        if len(observed) < 12:
            monthly_growth = 0.0015  # ~1.8% annual
            future_vals = observed.iloc[-1] * (1 + monthly_growth) ** np.arange(1, steps + 1)
        else:
            model = ExponentialSmoothing(
                observed,
                trend="add",
                damped_trend=True,
                seasonal=None,
                initialization_method="estimated",
            )
            fit = model.fit(optimized=True)
            future_vals = fit.forecast(steps).values

            # Guardrails: prevent unrealistic collapse or explosion
            last_val = observed.iloc[-1]
            floor = last_val * 0.90
            ceiling = last_val * 1.20
            future_vals = np.clip(future_vals, floor, ceiling)

        future_idx = pd.date_range(last_obs_date + pd.offsets.MonthBegin(1), pd.Timestamp(target_end), freq="MS")
        y.loc[future_idx] = future_vals
        out = y.ffill()

    result = pd.DataFrame({
        "date": full_idx,
        "ppi_value": out.values,
        "country_code": group["country_code"].iloc[0],
        "source": group["source"].iloc[0],
    })
    return result

In [ ]:
combined_ppi = (
    combined_ppi
    .sort_values(["country_code", "source", "date"])
    .groupby(["country_code", "source"], group_keys=False)
    .apply(extend_ppi_tail, target_end="2025-12-01")
    .reset_index(drop=True)
)

/tmp/ipykernel_1335/637060788.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(extend_ppi_tail, target_end="2025-12-01")


In [ ]:
combined_ppi.loc[lambda x: x.ppi_value.isna()]

,date,ppi_value,country_code,source


We now have a **combined_ppi** dataset in which the PPI ratios represent a valid time evolution from a common 2012 baseline reference for all coutnries.

All countries baseline is 2012 and all go up to 2025-12-01.

US no longer advantaged by having a 2005 base year

In [ ]:
combined_ppi.loc[lambda x: (x.source == 'FRED') & (x.country_code == 'OAC')].head()

,date,ppi_value,country_code,source
489,2012-06-01,100.0,OAC,FRED
490,2012-07-01,99.2,OAC,FRED
491,2012-08-01,99.1,OAC,FRED
492,2012-09-01,100.0,OAC,FRED
493,2012-10-01,100.1,OAC,FRED


## Building a suppler x date grid

## Restricting working panel to be from common month and year for all regions, onward

In [ ]:
combined_ppi_2012 = combined_ppi.loc[combined_ppi['date'] >= '2012-06-01'].copy()

In [ ]:
supplier_countries = list(baseline_circuits.keys())
all_dates = combined_ppi_2012['date'].drop_duplicates().sort_values()

supplier_grid = (
    all_dates.to_frame(name='date')
    .assign(key=1)
    .merge(
        pd.DataFrame({'country_code': supplier_countries, 'key': 1}),
        on='key'
    )
    .drop(columns='key')
)


In [ ]:
supplier_grid

,date,country_code
0,2012-06-01,USA
1,2012-06-01,JPN
2,2012-06-01,DEU
3,2012-06-01,FRA
4,2012-06-01,GBR
...,...,...
3255,2025-12-01,BRA
3256,2025-12-01,IND
3257,2025-12-01,THA
3258,2025-12-01,IDN


# Mapping suppliers to PPI series using ppi_map

In [ ]:
supplier_grid['ppi_source'] = supplier_grid['country_code'].map(lambda c: ppi_map[c][0])
supplier_grid['ppi_region'] = supplier_grid['country_code'].map(lambda c: ppi_map[c][1])


# Attaching regional PPI values to the supplier grid

In [ ]:
ppi_series = combined_ppi[[
    'date',
    'source',
    'country_code',
    'ppi_value'
]].rename(columns={
    'source': 'ppi_source',
    'country_code': 'ppi_region'
})

supplier_with_ppi = supplier_grid.merge(
    ppi_series,
    on=['date', 'ppi_source', 'ppi_region'],
    how='left'
)



In [ ]:
supplier_with_ppi

,date,country_code,ppi_source,ppi_region,ppi_value
0,2012-06-01,USA,FRED,USA,99.797898
1,2012-06-01,JPN,FRED,OAC,100.000000
2,2012-06-01,DEU,FRED,USA,99.797898
3,2012-06-01,FRA,FRED,USA,99.797898
4,2012-06-01,GBR,FRED,USA,99.797898
...,...,...,...,...,...
3255,2025-12-01,BRA,FRED,OAC,98.700000
3256,2025-12-01,IND,FRED,OAC,98.700000
3257,2025-12-01,THA,FRED,OAC,98.700000
3258,2025-12-01,IDN,FRED,OAC,98.700000


## Function to compute real_price any given product across each country and date combination

In [ ]:
def compute_real_price_row(row, baseline_dict):
    country = row['country_code']          # supplier country
    source = row['ppi_source']
    region = row['ppi_region']

    baseline = baseline_dict[country] # baseline price for country (2012)
    ppi_t = row['ppi_value'] # ppi at time t (month and year) for country
    ppi_anchor = ppi_anchor_2012[(source, region)] # base ppi value of fred data for [MEX, CAN, OAC, USA, or CHN]

    return baseline * (ppi_t / ppi_anchor)


In [ ]:
supplier_with_ppi['real_price'] = (
    supplier_with_ppi.apply(compute_real_price_row, axis=1, args=(baseline_circuits,))
)

In [ ]:
supplier_with_ppi = supplier_with_ppi.assign(
    product = 'integrated_circuit_components',
    year = lambda x: x.date.dt.year,
    month = lambda x: x.date.dt.month)

In [ ]:
supplier_with_ppi.loc[lambda x: x.ppi_value.isna()].country_code.unique()

array([], dtype=object)

In [ ]:
supplier_with_ppi.head()

,date,country_code,ppi_source,ppi_region,ppi_value,real_price,product,year,month
0,2012-06-01,USA,FRED,USA,99.797898,0.199596,integrated_circuit_components,2012,6
1,2012-06-01,JPN,FRED,OAC,100.000000,0.213043,integrated_circuit_components,2012,6
2,2012-06-01,DEU,FRED,USA,99.797898,0.199596,integrated_circuit_components,2012,6
3,2012-06-01,FRA,FRED,USA,99.797898,0.199596,integrated_circuit_components,2012,6
4,2012-06-01,GBR,FRED,USA,99.797898,0.199596,integrated_circuit_components,2012,6


In [ ]:
ic_components = supplier_with_ppi.copy()

In [ ]:
ic_components.head()

,date,country_code,ppi_source,ppi_region,ppi_value,real_price,product,year,month
0,2012-06-01,USA,FRED,USA,99.797898,0.199596,integrated_circuit_components,2012,6
1,2012-06-01,JPN,FRED,OAC,100.000000,0.213043,integrated_circuit_components,2012,6
2,2012-06-01,DEU,FRED,USA,99.797898,0.199596,integrated_circuit_components,2012,6
3,2012-06-01,FRA,FRED,USA,99.797898,0.199596,integrated_circuit_components,2012,6
4,2012-06-01,GBR,FRED,USA,99.797898,0.199596,integrated_circuit_components,2012,6


In [ ]:
ic_components.to_csv('ic_components_UPDATED.csv', index=False)

# Microprocessors/Microcontrollers

In [ ]:
# country-specific 2012 baseline wholesale unit price for mid-range microcontrollers
baseline_microprocessors_2012 = {
    'USA': 1.20, # Intel, AMD, TI, Freescale; high-cost domestic fabs
    'JPN': 1.32, # Renasas, Toshiba, Fujitsu; very high domestic fab costs
    'DEU': 1.28, # Infineon; high-cost EU fabs
    'FRA': 1.26, # STMicro; EU cost structure
    'GBR': 1.25, # ARM ecosystem; high engineering cost
    'NLD': 1.26, # NXP; EU fab/test cost structure
    'BEL': 1.26, # IMEC ecosystem; high R&D cost
    'CAN': 1.14, # Slightly lower overhead than US
    'AUS': 1.14, # Developed, smaller electronics base, similar to CAN
    'FIN': 1.25, # Nordic, high labor cost, niche electronics
    'SGP': 1.05, # Advanced fabs (GlobalFoundries), efficient operations
    'HKG': 1.00, # Regional hub; fabless ecosystem
    'MYS': 0.82, # Similar to MYS; strong electronics assembly
    'CHN': 0.76, # SMIC emergingl low labor; maturing fabs
    'MEX': 0.86, # Assembly/test; higher friction than Asia
    'BRA': 0.90, # Higher friction than Asia, but lower labor than US/EU; import heavy ecosystem
    'IND': 0.82, # Fabless ecosystem; low labor; limited fabs
    'IDN': 0.80, # Assembly/test; low labor
    'ARE': 1.22, # High-income, imported tech, niche fabs
    'THA': 0.82 # similar reasons
}

In [ ]:
len(baseline_microprocessors_2012.keys())

20

In [ ]:
supplier_with_ppi.head()

,date,country_code,ppi_source,ppi_region,ppi_value,real_price,product,year,month
0,2012-06-01,USA,FRED,USA,99.797898,0.199596,integrated_circuit_components,2012,6
1,2012-06-01,JPN,FRED,OAC,100.000000,0.213043,integrated_circuit_components,2012,6
2,2012-06-01,DEU,FRED,USA,99.797898,0.199596,integrated_circuit_components,2012,6
3,2012-06-01,FRA,FRED,USA,99.797898,0.199596,integrated_circuit_components,2012,6
4,2012-06-01,GBR,FRED,USA,99.797898,0.199596,integrated_circuit_components,2012,6


In [ ]:
supplier_with_ppi['real_price'] = (
    supplier_with_ppi.apply(compute_real_price_row, axis=1, args=(baseline_microprocessors_2012,))
)

In [ ]:
supplier_with_ppi = supplier_with_ppi.assign(
    product = 'microprocessors')

In [ ]:
supplier_with_ppi.head()

,date,country_code,ppi_source,ppi_region,ppi_value,real_price,product,year,month
0,2012-06-01,USA,FRED,USA,99.797898,1.197575,microprocessors,2012,6
1,2012-06-01,JPN,FRED,OAC,100.000000,1.339130,microprocessors,2012,6
2,2012-06-01,DEU,FRED,USA,99.797898,1.277413,microprocessors,2012,6
3,2012-06-01,FRA,FRED,USA,99.797898,1.257454,microprocessors,2012,6
4,2012-06-01,GBR,FRED,USA,99.797898,1.247474,microprocessors,2012,6


In [ ]:
microprocessors = supplier_with_ppi.copy()

In [ ]:
microprocessors.head()

,date,country_code,ppi_source,ppi_region,ppi_value,real_price,product,year,month
0,2012-06-01,USA,FRED,USA,99.797898,1.197575,microprocessors,2012,6
1,2012-06-01,JPN,FRED,OAC,100.000000,1.339130,microprocessors,2012,6
2,2012-06-01,DEU,FRED,USA,99.797898,1.277413,microprocessors,2012,6
3,2012-06-01,FRA,FRED,USA,99.797898,1.257454,microprocessors,2012,6
4,2012-06-01,GBR,FRED,USA,99.797898,1.247474,microprocessors,2012,6


In [ ]:
microprocessors.to_csv('microprocessors_UPDATED.csv', index=False)

# Transistors/Diodes

In [ ]:
baseline_transistors_2012 = {
    'USA': 0.14,
    'JPN': 0.14,
    'DEU': 0.14,
    'FRA': 0.14,
    'GBR': 0.14,
    'NLD': 0.14,
    'BEL': 0.14,
    'FIN': 0.14,
    'CAN': 0.13,
    'AUS': 0.13,
    'SGP': 0.12,
    'HKG': 0.11,
    'MYS': 0.10,
    'THA': 0.10,
    'CHN': 0.09,
    'MEX': 0.11,
    'BRA': 0.12,
    'IND': 0.10,
    'IDN': 0.10,
    'ARE': 0.14,
}


In [ ]:
len(baseline_transistors_2012.keys())

20

In [ ]:
supplier_with_ppi['real_price'] = (
    supplier_with_ppi.apply(compute_real_price_row, axis=1, args=(
        baseline_transistors_2012,
    )
))

In [ ]:
supplier_with_ppi = supplier_with_ppi.assign(
    product = 'transistors')

In [ ]:
supplier_with_ppi.head()

,date,country_code,ppi_source,ppi_region,ppi_value,real_price,product,year,month
0,2012-06-01,USA,FRED,USA,99.797898,0.139717,transistors,2012,6
1,2012-06-01,JPN,FRED,OAC,100.000000,0.142029,transistors,2012,6
2,2012-06-01,DEU,FRED,USA,99.797898,0.139717,transistors,2012,6
3,2012-06-01,FRA,FRED,USA,99.797898,0.139717,transistors,2012,6
4,2012-06-01,GBR,FRED,USA,99.797898,0.139717,transistors,2012,6


In [ ]:
transistors = supplier_with_ppi.copy()

In [ ]:
transistors.head()

,date,country_code,ppi_source,ppi_region,ppi_value,real_price,product,year,month
0,2012-06-01,USA,FRED,USA,99.797898,0.139717,transistors,2012,6
1,2012-06-01,JPN,FRED,OAC,100.000000,0.142029,transistors,2012,6
2,2012-06-01,DEU,FRED,USA,99.797898,0.139717,transistors,2012,6
3,2012-06-01,FRA,FRED,USA,99.797898,0.139717,transistors,2012,6
4,2012-06-01,GBR,FRED,USA,99.797898,0.139717,transistors,2012,6


In [ ]:
transistors.to_csv('transistors_UPDATED.csv', index=False)

# Power Devices

In [ ]:
baseline_power_devices_2012 = {
    'USA': 0.58,
    'JPN': 0.61,
    'DEU': 0.60,
    'FRA': 0.59,
    'GBR': 0.58,
    'NLD': 0.59,
    'BEL': 0.58,
    'CAN': 0.54,
    'AUS': 0.53,
    'FIN': 0.58,
    'SGP': 0.50,
    'HKG': 0.47,
    'MYS': 0.44,
    'CHN': 0.41,
    'MEX': 0.48,
    'BRA': 0.50,
    'IND': 0.45,
    'THA': 0.44,
    'IDN': 0.43,
    'ARE': 0.56,
}


In [ ]:
len(baseline_power_devices_2012.keys())

20

In [ ]:
supplier_with_ppi.head()

,date,country_code,ppi_source,ppi_region,ppi_value,real_price,product,year,month
0,2012-06-01,USA,FRED,USA,99.797898,0.139717,transistors,2012,6
1,2012-06-01,JPN,FRED,OAC,100.000000,0.142029,transistors,2012,6
2,2012-06-01,DEU,FRED,USA,99.797898,0.139717,transistors,2012,6
3,2012-06-01,FRA,FRED,USA,99.797898,0.139717,transistors,2012,6
4,2012-06-01,GBR,FRED,USA,99.797898,0.139717,transistors,2012,6


In [ ]:
supplier_with_ppi['real_price'] = (
    supplier_with_ppi.apply(compute_real_price_row, axis=1, args=(
        baseline_power_devices_2012,
    )))

In [ ]:
supplier_with_ppi.assign(
    product = 'power_devices').head()

,date,country_code,ppi_source,ppi_region,ppi_value,real_price,product,year,month
0,2012-06-01,USA,FRED,USA,99.797898,0.578828,power_devices,2012,6
1,2012-06-01,JPN,FRED,OAC,100.000000,0.618841,power_devices,2012,6
2,2012-06-01,DEU,FRED,USA,99.797898,0.598787,power_devices,2012,6
3,2012-06-01,FRA,FRED,USA,99.797898,0.588808,power_devices,2012,6
4,2012-06-01,GBR,FRED,USA,99.797898,0.578828,power_devices,2012,6


In [ ]:
power_devices = supplier_with_ppi.assign(
    product = 'power_devices').copy()

In [ ]:
power_devices.head()

,date,country_code,ppi_source,ppi_region,ppi_value,real_price,product,year,month
0,2012-06-01,USA,FRED,USA,99.797898,0.578828,power_devices,2012,6
1,2012-06-01,JPN,FRED,OAC,100.000000,0.618841,power_devices,2012,6
2,2012-06-01,DEU,FRED,USA,99.797898,0.598787,power_devices,2012,6
3,2012-06-01,FRA,FRED,USA,99.797898,0.588808,power_devices,2012,6
4,2012-06-01,GBR,FRED,USA,99.797898,0.578828,power_devices,2012,6


In [ ]:
power_devices.to_csv('power_devices_UPDATED.csv', index=False)

In [ ]:
power_devices.real_price.info()

<class 'pandas.core.series.Series'>
RangeIndex: 3260 entries, 0 to 3259
Series name: real_price
Non-Null Count  Dtype  
--------------  -----  
3260 non-null   float64
dtypes: float64(1)
memory usage: 25.6 KB


# Commodity Prices

In [4]:
commodity_prices = commodity_prices.loc[lambda x: x.date >= '2012-06-01']

In [5]:
commodity_prices.to_csv('commodity_prices_UPDATED.csv', index=False)

# Combining Products into Single Dataset

In [ ]:
ic_components.head()

,date,country_code,ppi_source,ppi_region,ppi_value,real_price,product,year,month
0,2012-06-01,USA,FRED,USA,99.797898,0.199596,integrated_circuit_components,2012,6
1,2012-06-01,JPN,FRED,OAC,100.000000,0.213043,integrated_circuit_components,2012,6
2,2012-06-01,DEU,FRED,USA,99.797898,0.199596,integrated_circuit_components,2012,6
3,2012-06-01,FRA,FRED,USA,99.797898,0.199596,integrated_circuit_components,2012,6
4,2012-06-01,GBR,FRED,USA,99.797898,0.199596,integrated_circuit_components,2012,6


In [ ]:
microprocessors.head()

,date,country_code,ppi_source,ppi_region,ppi_value,real_price,product,year,month
0,2012-06-01,USA,FRED,USA,99.797898,1.197575,microprocessors,2012,6
1,2012-06-01,JPN,FRED,OAC,100.000000,1.339130,microprocessors,2012,6
2,2012-06-01,DEU,FRED,USA,99.797898,1.277413,microprocessors,2012,6
3,2012-06-01,FRA,FRED,USA,99.797898,1.257454,microprocessors,2012,6
4,2012-06-01,GBR,FRED,USA,99.797898,1.247474,microprocessors,2012,6


In [ ]:
transistors.head()

,date,country_code,ppi_source,ppi_region,ppi_value,real_price,product,year,month
0,2012-06-01,USA,FRED,USA,99.797898,0.139717,transistors,2012,6
1,2012-06-01,JPN,FRED,OAC,100.000000,0.142029,transistors,2012,6
2,2012-06-01,DEU,FRED,USA,99.797898,0.139717,transistors,2012,6
3,2012-06-01,FRA,FRED,USA,99.797898,0.139717,transistors,2012,6
4,2012-06-01,GBR,FRED,USA,99.797898,0.139717,transistors,2012,6


In [ ]:
power_devices.head()

,date,country_code,ppi_source,ppi_region,ppi_value,real_price,product,year,month
0,2012-06-01,USA,FRED,USA,99.797898,0.578828,power_devices,2012,6
1,2012-06-01,JPN,FRED,OAC,100.000000,0.618841,power_devices,2012,6
2,2012-06-01,DEU,FRED,USA,99.797898,0.598787,power_devices,2012,6
3,2012-06-01,FRA,FRED,USA,99.797898,0.588808,power_devices,2012,6
4,2012-06-01,GBR,FRED,USA,99.797898,0.578828,power_devices,2012,6


In [ ]:
combined_products = pd.concat([
    ic_components, microprocessors, transistors, power_devices
])

In [ ]:
combined_products

,date,country_code,ppi_source,ppi_region,ppi_value,real_price,product,year,month
0,2012-06-01,USA,FRED,USA,99.797898,0.199596,integrated_circuit_components,2012,6
1,2012-06-01,JPN,FRED,OAC,100.000000,0.213043,integrated_circuit_components,2012,6
2,2012-06-01,DEU,FRED,USA,99.797898,0.199596,integrated_circuit_components,2012,6
3,2012-06-01,FRA,FRED,USA,99.797898,0.199596,integrated_circuit_components,2012,6
4,2012-06-01,GBR,FRED,USA,99.797898,0.199596,integrated_circuit_components,2012,6
...,...,...,...,...,...,...,...,...,...
3255,2025-12-01,BRA,FRED,OAC,98.700000,0.500652,power_devices,2025,12
3256,2025-12-01,IND,FRED,OAC,98.700000,0.450587,power_devices,2025,12
3257,2025-12-01,THA,FRED,OAC,98.700000,0.440574,power_devices,2025,12
3258,2025-12-01,IDN,FRED,OAC,98.700000,0.430561,power_devices,2025,12


In [ ]:
combined_products.to_csv('combined_products_UPDATED.csv', index=False)